# Your first scraper
In this project, we will guide you step by step through the process of:

1. creating a self-contained development environment.
1. retrieving some information from an API (a website for computers)
2. leveraging it to scrape a website that does not provide an API
3. saving the output for later processing

Here we query an API for a list of countries and their past leaders. We then extract and sanitize their short bio from Wikipedia. Finally, we save the data to disk.

This task is often the first (coding) step of a datascience project and you will often come back to it in the future.

You will study topics such as *scraping*, *data structures*, *regular expressions*, *concurrency* and *file handling*. We will point out useful resources at the appropriate time. 

Let's dive in!

## 0. Creating a clean environment

Use the [`venv`](https://docs.python.org/3/library/venv.html) command to create a new environment called `wikipedia_scraper_env`.

Activate it and add it to you `.gitignore` file. 

You will find more info about virtual environments in the course content and on the web.

## 1. API Scraping

### 1a. A simple API query
You will start with the basics: how to do a simple request to an [API endpoint](../../2.python/2.python_advanced/05.Scraping/5.apis.ipynb).

You will use the [requests](https://requests.readthedocs.io/en/latest/) external library through the `import` keyword. NOTE: external libraries need to be installed first. Check the [request Quickstart](https://requests.readthedocs.io/en/latest/user/quickstart/) section of the documentation to:

1. Use the `get()` method to connect to this endpoint: https://country-leaders.onrender.com/status
2. Check if the `status_code` is equal to 200, which means OK.
    * if OK, `print()` the `text`` of the response.
    * if not, `print()` the `status_code`. 

Here is an explanation of [HTTP status codes](https://en.wikipedia.org/wiki/List_of_HTTP_status_codes).


In [4]:
# import the requests library (1 line)
import requests

# assign the root url (without /status) to the root_url variable for ease of reference (1 line)
root_url = "https://country-leaders.onrender.com"

# assign the /status endpoint to another variable called status_url (1 line)
status_url = "status"

# query the /status endpoint using the get() method and store it in the req variable (1 line)
req = requests.get(f"{root_url}/{status_url}")

# check the status_code using a condition and print appropriate messages (4 lines)
if req.status_code == 200:
    print(req.text)
else:
    print(req.status_code)

"Alive"


### 1b. Dealing with JSON

[JSON](https://quickref.me/json) is the preferred format to deal with data over the web. You cannot avoid it so you would better get acquainted.

Connect to another endpoint called `/countries` but this time the API will return data in the JSON format. 


In [5]:
# Set the countries_url variable (1 line)
countries_url = "countries"

# query the /countries endpoint using the get() method and store it in the req variable (1 line)
req = requests.get(f"{root_url}/{countries_url}")

# Get the JSON content and store it in the countries variable (1 line)
countries = req.json()

# display the request's status code and the countries variable (1 line)
print(req.status_code, countries)


403 {'message': 'The cookie is missing'}


### 1c. Cookies anyone?

It looks like the access to this API is restricted...
Query the `/cookie` endpoint and extract the appropriate field to access your cookie.

You will need to use this cookie in each of the following API requests.

In [6]:
# Set the cookie_url variable (1 line)
cookie_url = "cookie"

# Query the enpoint, set the cookies variable and display it (2 lines)
cookie = requests.get(f"{root_url}/{cookie_url}")
print(cookie.headers)

{'Date': 'Thu, 04 Jun 2026 09:16:20 GMT', 'Content-Type': 'application/json', 'Transfer-Encoding': 'chunked', 'Connection': 'keep-alive', 'cf-cache-status': 'DYNAMIC', 'rndr-id': 'ee3b68fb-a88f-42fe', 'Server': 'cloudflare', 'Set-Cookie': 'user_cookie=71836732-993c-40b1-b110-b41e88b9153a; Path=/; SameSite=lax', 'vary': 'Accept-Encoding', 'x-render-origin-server': 'uvicorn', 'CF-RAY': 'a065d692aecb25aa-BRU', 'alt-svc': 'h3=":443"; ma=86400'}


Try to query the countries endpoint using the cookie, save the output and print it.

In [4]:
# query the /countries endpoint, assign the output to the countries variable (1 line)
countries = requests.get(f"{root_url}/{countries_url}",cookies=cookie.cookies)

# display the countries variable (1 line)
print(countries.text)


["fr","us","be","ma","ru"]


Chances are the cookie has expired... Thanksfully, you got a nice error message. For now, simply execute the last 2 cells quickly so you get a result.

### 1d. Getting the actual data from the API

Query the `/leaders` endpoint.

In [5]:
# Set the leaders_url variable (1 line)
leaders_url = "leaders"

# query the /leaders endpoint, assign the output to the leaders variable (1 line)
leaders = requests.get(f"{root_url}/{leaders_url}",cookies=cookie.cookies)

# display the leaders variable (1 line)
print(leaders.json())


{'message': 'Please specify a country'}


It looks like this endpoint requires additional information in order to return its result. Check the API [*documentation*](https://country-leaders.onrender.com/docs) in your web browser.

Change the query to accept *parameters*. You should know where to find help by now.

In [6]:
# query the /leaders endpoint using cookies and parameters (take any country in countries)
# assign the output to the leaders variable (1 line)
leaders = requests.get(f"{root_url}/{leaders_url}",cookies=cookie.cookies,params={"country":"ma"})

# display the leaders variable (1 line)
print(leaders.text)

[{"id":"Q57553","first_name":"Mohammed","last_name":"None","birth_date":"1963-08-21","death_date":null,"place_of_birth":"Rabat","wikipedia_url":"https://ar.wikipedia.org/wiki/%D9%85%D8%AD%D9%85%D8%AF_%D8%A7%D9%84%D8%B3%D8%A7%D8%AF%D8%B3_%D8%A8%D9%86_%D8%A7%D9%84%D8%AD%D8%B3%D9%86","start_mandate":"1999-07-23","end_mandate":null},{"id":"Q69103","first_name":"Hassan","last_name":"None","birth_date":"1929-07-09","death_date":"1999-07-20","place_of_birth":"Rabat","wikipedia_url":"https://ar.wikipedia.org/wiki/%D8%A7%D9%84%D8%AD%D8%B3%D9%86_%D8%A7%D9%84%D8%AB%D8%A7%D9%86%D9%8A_%D8%A8%D9%86_%D9%85%D8%AD%D9%85%D8%AF","start_mandate":"1961-02-26","end_mandate":"1999-07-23"},{"id":"Q193874","first_name":"Mohammed","last_name":"None","birth_date":"1909-08-10","death_date":"1961-02-26","place_of_birth":"Fez","wikipedia_url":"https://ar.wikipedia.org/wiki/%D9%85%D8%AD%D9%85%D8%AF_%D8%A7%D9%84%D8%AE%D8%A7%D9%85%D8%B3_%D8%A8%D9%86_%D9%8A%D9%88%D8%B3%D9%81","start_mandate":"1957-08-14","end_mandate":

### 1e. A sneak peak at the data (finally)

Look inside a few examples. Notice the dictionary keys available for each entry. You have your first example of *structured data*. This data was sanitized for your benefit, meaning it is readily exploitable without modification.

You will also notice there is a Wikipedia link for each entry. You will need to extract additional information there. This will be a case of *semi-structured* data.

The /countries endpoint returns a `list` of several country codes.

You need to loop through this list and query the /leaders endpoint for each one. Save each `json` result in a dictionary called `leaders_per_country`.

In [7]:
# 4 lines
leader_per_country = {}
for coun in countries.json():
    leader = requests.get(f"{root_url}/{leaders_url}", cookies=cookie.cookies, params={"country":coun})
    leader_per_country[coun] = leader.json()
print(leader_per_country)

{'fr': [{'id': 'Q157', 'first_name': 'François', 'last_name': 'Hollande', 'birth_date': '1954-08-12', 'death_date': None, 'place_of_birth': 'Rouen', 'wikipedia_url': 'https://fr.wikipedia.org/wiki/Fran%C3%A7ois_Hollande', 'start_mandate': '2012-05-15', 'end_mandate': '2017-05-14'}, {'id': 'Q329', 'first_name': 'Nicolas', 'last_name': 'Sarkozy', 'birth_date': '1955-01-28', 'death_date': None, 'place_of_birth': 'Paris', 'wikipedia_url': 'https://fr.wikipedia.org/wiki/Nicolas_Sarkozy', 'start_mandate': '2007-05-16', 'end_mandate': '2012-05-15'}, {'id': 'Q2038', 'first_name': 'François', 'last_name': 'Mitterrand', 'birth_date': '1916-10-26', 'death_date': '1996-01-08', 'place_of_birth': 'Jarnac', 'wikipedia_url': 'https://fr.wikipedia.org/wiki/Fran%C3%A7ois_Mitterrand', 'start_mandate': '1981-05-21', 'end_mandate': '1995-05-17'}, {'id': 'Q2042', 'first_name': 'Charles', 'last_name': 'de Gaulle', 'birth_date': '1890-11-22', 'death_date': '1970-11-09', 'place_of_birth': 'Lille', 'wikipedia_u

In [8]:
# or 1 line


It is finally time to create a `get_leaders()` function for the above code. You will build on it later-on. This function takes no parameter. Inside it, you will need to:
1. define the urls
2. get the cookies
2. get the countries
3. loop over them and save their leaders in a dictionary
4. return the dictionary

In [9]:
# < 15 lines
def get_leaders():
    root_url = "https://country-leaders.onrender.com"
    countries_url = "countries"
    cookie_url = "cookie"
    leaders_url = "leaders"

    cookie = requests.get(f"{root_url}/{cookie_url}")
    countries = requests.get(f"{root_url}/{countries_url}",cookies=cookie.cookies)
    leader_per_country = {}
    for coun in countries.json():
        leader = requests.get(f"{root_url}/{leaders_url}", cookies=cookie.cookies, params={"country":coun})
        leader_per_country[coun] = leader.json()
    return leader_per_country

Test your function, save the result in the `leaders_per_country` dictionary and check its ouput.

In [10]:
# 2 lines
leaders_per_country = get_leaders()
leaders_per_country

{'fr': [{'id': 'Q157',
   'first_name': 'François',
   'last_name': 'Hollande',
   'birth_date': '1954-08-12',
   'death_date': None,
   'place_of_birth': 'Rouen',
   'wikipedia_url': 'https://fr.wikipedia.org/wiki/Fran%C3%A7ois_Hollande',
   'start_mandate': '2012-05-15',
   'end_mandate': '2017-05-14'},
  {'id': 'Q329',
   'first_name': 'Nicolas',
   'last_name': 'Sarkozy',
   'birth_date': '1955-01-28',
   'death_date': None,
   'place_of_birth': 'Paris',
   'wikipedia_url': 'https://fr.wikipedia.org/wiki/Nicolas_Sarkozy',
   'start_mandate': '2007-05-16',
   'end_mandate': '2012-05-15'},
  {'id': 'Q2038',
   'first_name': 'François',
   'last_name': 'Mitterrand',
   'birth_date': '1916-10-26',
   'death_date': '1996-01-08',
   'place_of_birth': 'Jarnac',
   'wikipedia_url': 'https://fr.wikipedia.org/wiki/Fran%C3%A7ois_Mitterrand',
   'start_mandate': '1981-05-21',
   'end_mandate': '1995-05-17'},
  {'id': 'Q2042',
   'first_name': 'Charles',
   'last_name': 'de Gaulle',
   'birth_d

## 2. Extracting data from Wikipedia

Query one of the leaders' Wikipedia urls and display its `text` (not JSON).

In [7]:
# 3 lines
url = "https://fr.wikipedia.org/wiki/Adolphe_Thiers"
headers = {"User-Agent":"Mozilla/5.0"}
response = requests.get(url,headers=headers)
#print(response.text)

Ouch! You get the raw HTML code of the webpage. If you try to deal with it without tools, you will be there all night. Instead, use the [beautiful soup 4](https://www.crummy.com/software/BeautifulSoup/bs4/doc/) *external* library. You will find more info about it [here](../../2.python/2.python_advanced/05.Scraping/1.beautifulsoup_basic.ipynb) and [here](../../2.python/2.python_advanced/05.Scraping/2.beautifulsoup_advanced.ipynb)

Using the Quickstart section, start by importing the library and loading the output of your `get_text()` function.

Use the `prettify()` function and print it to take a look. You will start the actual parsing in the next step.

In [8]:
# 3 lines
from bs4 import BeautifulSoup
soup = BeautifulSoup(response.content, "html")
#print(soup.prettify())

That looks better but you need to extract the right part of the webpage: the text of the first paragraph.

It is a bit tricky because Wikipedia pages slightly differ in structure from one language to the next. We cannot simply get the text for the first HTML paragraph.

You will start by getting all the HTML paragraphs from the HTML source and saving them in the `paragraphs` variable.

Use the documentation or google the appropriate keywords.

In [13]:
# 2 lines
paragraphs = [par for par in soup.find_all("p")]
paragraphs

[<p>Pour les articles homonymes, voir <a class="mw-disambig" href="/wiki/Thiers_(homonymie)" title="Thiers (homonymie)">Thiers (homonymie)</a>.
 </p>,
 <p>Cet article concerne l'homme d'État. Pour l'architecte, voir <a href="/wiki/Adolphe_Thiers_(architecte)" title="Adolphe Thiers (architecte)">Adolphe Thiers (architecte)</a>.
 </p>,
 <p class="mw-empty-elt">
 </p>,
 <p><b>Adolphe Thiers</b>, né le <time class="nowrap bday" data-sort-value="1797-04-15" datetime="1797-04-15">15 avril 1797</time> (<span class="nowrap"><a href="/wiki/26_germinal" title="26 germinal">26</a> <a href="/wiki/Germinal" title="Germinal">germinal</a> <a href="/wiki/An_V" title="An V">an <abbr class="abbr" title="5">V</abbr></a></span>) à <a href="/wiki/Marseille" title="Marseille">Marseille</a> (Bouches-du-Rhône) et mort le <time class="nowrap dday" data-sort-value="1877-09-03" datetime="1877-09-03">3 septembre 1877</time> à <a href="/wiki/Saint-Germain-en-Laye" title="Saint-Germain-en-Laye">Saint-Germain-en-Lay

If you try different urls, you might find that the paragraph you want may be at a different index each time.

That is where you need to be clever and ask yourself what would be a reliable way to identify the right index ie. which string matches only the first paragraph whatever the language...

Spend a good 30 minutes on the problem and brainstorm with your fellow learners. If you come out empty handed, ask your coach.

1. Loop over the HTML paragraphs
2. When you have identified the correct one:
   * Store the [text](https://www.crummy.com/software/BeautifulSoup/bs4/doc/#output) inside the `first_paragraph` variable
   * Exit the loop

In [14]:
# <10 lines
first_paragraph = paragraphs[0]
for par in paragraphs:
    if str(par).startswith("<p><b>"):
        first_paragraph = par
        break
print(type(first_paragraph))
print(first_paragraph.text)


<class 'bs4.element.Tag'>
Adolphe Thiers, né le 15 avril 1797 (26 germinal an V) à Marseille (Bouches-du-Rhône) et mort le 3 septembre 1877 à Saint-Germain-en-Laye (Seine-et-Oise), est un avocat, journaliste, historien et homme d'État français.



At this stage, you can create a function to maintain consistency in your code. We will give you its *skeleton*, you will copy the code you wrote and make it work inside a function.

Don't forget to test your function.

In [ ]:
# 10 lines
def get_first_paragraph(wikipedia_url : str) -> str:
    print(wikipedia_url) # keep this for the rest of the notebook
    headers = {"User-Agent":"Mozilla/5.0"}
    response = requests.get(wikipedia_url,headers=headers)
    soup = BeautifulSoup(response.content, "html")
    paragraphs = [par for par in soup.find_all("p")]
    for par in paragraphs:
        if str(par).startswith("<p><b>"):
            return par.text
    return ""

In [ ]:
# Test: 3 lines
wikipedia_url = "https://fr.wikipedia.org/wiki/Fran%C3%A7ois_Hollande"
first_paragraph = get_first_paragraph(wikipedia_url)
print(first_paragraph)

https://fr.wikipedia.org/wiki/Fran%C3%A7ois_Hollande


François Hollande [fʁɑ̃swa ɔlɑ̃d][n 3] Écouterⓘ, né le 12 août 1954 à Rouen (Seine-Inférieure), est un haut fonctionnaire et homme d'État français. Il est président de la République française du 15 mai 2012 au 14 mai 2017.



### 2a. Regular expressions to the rescue

Now that you have extracted the content of the first paragraph, the only thing that remains to finish your Wikipedia scraper is to sanitize the output.

Indeed some Wikipedia references, HTML code, phonetic pronunciation etc. may linger. You might find *regular expressions* handy to get rid of them and obtain pristine text. You will find some useful documentation about regular expressions [here](../../2.python/2.python_advanced/03.Regex/regex.ipynb)

Once you have one of your regex working online, try it in the cell below. 

Hints: 
* Check the `sub()` method documentation.
* Make sure to test urls in different languages. Some may look good but other do not.

In [ ]:
# 3 lines
import re
result1 = re.sub(r'[^a-zA-Z0-9 ,éèû\'\-]',"",str(first_paragraph.text))
result2 = ''.join([char for char in str(first_paragraph.text) if (char.isalnum() or char in " -'.,()")])
print(result1)
print(result2)

def sanitize(text : str) -> str:
    return re.sub(r'[^a-zA-Z0-9 ,éèû\'\-]',"",text)


Franois Hollande fswa ldn 3 couter, né le 12 août 1954  Rouen Seine-Inférieure, est un haut fonctionnaire et homme d'tat franais Il est président de la République franaise du 15 mai 2012 au 14 mai 2017
François Hollande fʁɑswa ɔlɑdn 3 Écouter, né le 12 août 1954 à Rouen (Seine-Inférieure), est un haut fonctionnaire et homme d'État français. Il est président de la République française du 15 mai 2012 au 14 mai 2017.


Overwrite the `get_first_paragraph()` function by applying your regex to the first paragraph before returning it.

In [ ]:
# 10 lines
def get_first_paragraph(wikipedia_url : str) -> str:
    print(wikipedia_url) # keep this for the rest of the notebook
    headers = {"User-Agent":"Mozilla/5.0"}
    response = requests.get(wikipedia_url,headers=headers)
    soup = BeautifulSoup(response.content, "html")
    paragraphs = [par for par in soup.find_all("p")]
    for par in paragraphs:
        if str(par).startswith("<p><b>"):
            return sanitize(par.text)
    return ""

Come up with other regexes to capture other patterns and sanitize the outputs completely. Modify your `get_first_paragraph()` function accordingly.

In [19]:
# < 20 lines


## 3. Putting it all together

Let's go back to your `get_leaders()` function and update it with an *inner* loop over each leader. You will query the url provided and extract the first paragraph using the `get_first_paragraph()` function you just finished. You will then update that `leader`'s dictionary and move on to the next one.

Notice, the rest of the code should not change since you modify the leader's data one by one.

In [15]:
# < 20 lines
def get_leaders():
    root_url = "https://country-leaders.onrender.com"
    countries_url = "countries"
    cookie_url = "cookie"
    leaders_url = "leaders"

    cookie = requests.get(f"{root_url}/{cookie_url}")
    countries = requests.get(f"{root_url}/{countries_url}",cookies=cookie.cookies)
    leaders_per_country = {}
    for coun in countries.json():
        leaders = requests.get(f"{root_url}/{leaders_url}", cookies=cookie.cookies, params={"country":coun})
        for leader in leaders.json():
            leader["bio"] = get_first_paragraph(leader["wikipedia_url"])
        leaders_per_country[coun] = leaders.json()
        
    return leaders_per_country

In [16]:
# Check the output of your function (2 lines)
leaders_per_country = get_leaders()
leaders_per_country

https://fr.wikipedia.org/wiki/Fran%C3%A7ois_Hollande


KeyboardInterrupt: 

Does the function crash in the middle of the loop? Chances are the cookies have expired while looping over the leaders.

Modify your function with an *exception* or check if the `status_code` is a cookie error. In either case, get new cookies and query the api again.

If your code did not crash,

In [ ]:
# < 25 lines
def get_leaders():
    root_url = "https://country-leaders.onrender.com"
    countries_url = "countries"
    cookie_url = "cookie"
    leaders_url = "leaders"

    cookie = requests.get(f"{root_url}/{cookie_url}")
    countries = requests.get(f"{root_url}/{countries_url}",cookies=cookie.cookies)
    leaders_per_country = {}
    for coun in countries.json():
        leaders = requests.get(f"{root_url}/{leaders_url}", cookies=cookie.cookies, params={"country":coun})
        if leaders.status_code != 200:
            cookie = requests.get(f"{root_url}/{cookie_url}")
            leaders = requests.get(f"{root_url}/{leaders_url}", cookies=cookie.cookies, params={"country":coun})
        for leader in leaders.json():
            leader["bio"] = get_first_paragraph(leader["wikipedia_url"])
        leaders_per_country[coun] = leaders.json()
        
    return leaders_per_country


Check the output of your function again.

In [ ]:
# Check the output of your function (1 line)
leaders_per_country = get_leaders()
leaders_per_country

cookie: <RequestsCookieJar[<Cookie user_cookie=55da8578-7aa3-406e-92fa-7f49df56b976 for country-leaders.onrender.com/>]>
200 <Response [200]>
[{'id': 'Q157', 'first_name': 'François', 'last_name': 'Hollande', 'birth_date': '1954-08-12', 'death_date': None, 'place_of_birth': 'Rouen', 'wikipedia_url': 'https://fr.wikipedia.org/wiki/Fran%C3%A7ois_Hollande', 'start_mandate': '2012-05-15', 'end_mandate': '2017-05-14'}, {'id': 'Q329', 'first_name': 'Nicolas', 'last_name': 'Sarkozy', 'birth_date': '1955-01-28', 'death_date': None, 'place_of_birth': 'Paris', 'wikipedia_url': 'https://fr.wikipedia.org/wiki/Nicolas_Sarkozy', 'start_mandate': '2007-05-16', 'end_mandate': '2012-05-15'}, {'id': 'Q2038', 'first_name': 'François', 'last_name': 'Mitterrand', 'birth_date': '1916-10-26', 'death_date': '1996-01-08', 'place_of_birth': 'Jarnac', 'wikipedia_url': 'https://fr.wikipedia.org/wiki/Fran%C3%A7ois_Mitterrand', 'start_mandate': '1981-05-21', 'end_mandate': '1995-05-17'}, {'id': 'Q2042', 'first_name

Well done! It took a while however... Let's speed things up. The main *bottleneck* is the loop. We call on the Wikipedia website many times.

You will use the same *session* to call all the wikipedia pages. Check the *Advanced Usage* section of the Requests module's documentation.

Start by modifying the `get_first_paragraph()` function to accept a session parameter and adjust the `get()` method call.

In [ ]:
# < 20 lines
from requests import Session

def get_first_paragraph(wikipedia_url : str, session : Session) -> str:
    print(wikipedia_url) # keep this for the rest of the notebook
    headers = {"User-Agent":"Mozilla/5.0"}
    response = session.get(wikipedia_url,headers=headers)
    soup = BeautifulSoup(response.content, "html")
    paragraphs = soup.find_all("p")
    for par in paragraphs:
        if str(par).startswith("<p><b>"):
            return sanitize(par.text)
    return ""

Modify your `get_leaders()` function to make use of a single session for all the Wikipedia calls.
1. create a `Session` object outside of the loop over countries.
2. pass it to the `get_first_paragraph()` function as an argument.

In [ ]:
# <25 lines
def get_leaders():
    root_url = "https://country-leaders.onrender.com"
    countries_url = "countries"
    cookie_url = "cookie"
    leaders_url = "leaders"

    cookie = requests.get(f"{root_url}/{cookie_url}")
    countries = requests.get(f"{root_url}/{countries_url}",cookies=cookie.cookies)
    leaders_per_country = {}
    with Session() as session:
        for coun in countries.json():
            leaders = requests.get(f"{root_url}/{leaders_url}", cookies=cookie.cookies, params={"country":coun})
            if leaders.status_code != 200:
                cookie = requests.get(f"{root_url}/{cookie_url}")
                leaders = requests.get(f"{root_url}/{leaders_url}", cookies=cookie.cookies, params={"country":coun})
            leaders_with_bio = []
            for leader in leaders.json():
                leader["bio"] = get_first_paragraph(leader["wikipedia_url"], session)
                leaders_with_bio.append(leader)
            leaders_per_country[coun] = leaders_with_bio
        
    return leaders_per_country

Test your new functions.



In [21]:
leaders_per_country = get_leaders()
leaders_per_country

https://fr.wikipedia.org/wiki/Fran%C3%A7ois_Hollande
https://fr.wikipedia.org/wiki/Nicolas_Sarkozy
https://fr.wikipedia.org/wiki/Fran%C3%A7ois_Mitterrand
https://fr.wikipedia.org/wiki/Charles_de_Gaulle
https://fr.wikipedia.org/wiki/Jacques_Chirac
https://fr.wikipedia.org/wiki/Val%C3%A9ry_Giscard_d%27Estaing
https://fr.wikipedia.org/wiki/Georges_Pompidou
https://fr.wikipedia.org/wiki/Adolphe_Thiers
https://fr.wikipedia.org/wiki/Napol%C3%A9on_III
https://fr.wikipedia.org/wiki/Paul_Doumer
https://fr.wikipedia.org/wiki/Alain_Poher
https://fr.wikipedia.org/wiki/Albert_Lebrun
https://fr.wikipedia.org/wiki/Ren%C3%A9_Coty
https://fr.wikipedia.org/wiki/Vincent_Auriol
https://fr.wikipedia.org/wiki/Patrice_de_Mac_Mahon
https://fr.wikipedia.org/wiki/%C3%89mile_Loubet
https://fr.wikipedia.org/wiki/Raymond_Poincar%C3%A9
https://fr.wikipedia.org/wiki/Sadi_Carnot_(homme_d%27%C3%89tat)
https://fr.wikipedia.org/wiki/Alexandre_Millerand
https://fr.wikipedia.org/wiki/Gaston_Doumergue
https://fr.wikipedia.

{'fr': [{'id': 'Q157',
   'first_name': 'François',
   'last_name': 'Hollande',
   'birth_date': '1954-08-12',
   'death_date': None,
   'place_of_birth': 'Rouen',
   'wikipedia_url': 'https://fr.wikipedia.org/wiki/Fran%C3%A7ois_Hollande',
   'start_mandate': '2012-05-15',
   'end_mandate': '2017-05-14',
   'bio': "François Hollande [fʁɑ̃swa ɔlɑ̃d][n 3] Écouterⓘ, né le 12 août 1954 à Rouen (Seine-Inférieure), est un haut fonctionnaire et homme d'État français. Il est président de la République française du 15 mai 2012 au 14 mai 2017.\n"},
  {'id': 'Q329',
   'first_name': 'Nicolas',
   'last_name': 'Sarkozy',
   'birth_date': '1955-01-28',
   'death_date': None,
   'place_of_birth': 'Paris',
   'wikipedia_url': 'https://fr.wikipedia.org/wiki/Nicolas_Sarkozy',
   'start_mandate': '2007-05-16',
   'end_mandate': '2012-05-15',
   'bio': "Nicolas Sarközy de Nagy-Bocsa, dit Nicolas Sarkozy (/ni.kɔ.la saʁ.kɔ.zi/[d] Écouterⓘ\xa0; en hongrois Sárközy ou Sárközi [ˈʃaːɾkøzi][3],[4],[5]), né le 2

## 4. Saving your hard work

The final step is to save the ``leaders_per_country`` dictionary in the `leaders.json` file using the [json](https://docs.python.org/3/library/json.html) module. Check out the `with` statement.

In [22]:
# 3 lines
import json
with open("leaders.json","w") as file:
    json.dump(leaders_per_country, file)

Make sure the file can be read back. Write the code to read the file. And check the variables are the same.

In [23]:
# 3 lines
with open("leaders.json","r") as file :
    data = json.load(file)
data

{'fr': [{'id': 'Q157',
   'first_name': 'François',
   'last_name': 'Hollande',
   'birth_date': '1954-08-12',
   'death_date': None,
   'place_of_birth': 'Rouen',
   'wikipedia_url': 'https://fr.wikipedia.org/wiki/Fran%C3%A7ois_Hollande',
   'start_mandate': '2012-05-15',
   'end_mandate': '2017-05-14',
   'bio': "François Hollande [fʁɑ̃swa ɔlɑ̃d][n 3] Écouterⓘ, né le 12 août 1954 à Rouen (Seine-Inférieure), est un haut fonctionnaire et homme d'État français. Il est président de la République française du 15 mai 2012 au 14 mai 2017.\n"},
  {'id': 'Q329',
   'first_name': 'Nicolas',
   'last_name': 'Sarkozy',
   'birth_date': '1955-01-28',
   'death_date': None,
   'place_of_birth': 'Paris',
   'wikipedia_url': 'https://fr.wikipedia.org/wiki/Nicolas_Sarkozy',
   'start_mandate': '2007-05-16',
   'end_mandate': '2012-05-15',
   'bio': "Nicolas Sarközy de Nagy-Bocsa, dit Nicolas Sarkozy (/ni.kɔ.la saʁ.kɔ.zi/[d] Écouterⓘ\xa0; en hongrois Sárközy ou Sárközi [ˈʃaːɾkøzi][3],[4],[5]), né le 2

Make a function `save(leaders_per_country)` to call this code easily.

In [24]:
# 3 lines
def save(leaders_per_country : dict):
    with open("leaders.json","w") as file:
        json.dump(leaders_per_country, file)

In [25]:
# Call the function (1 line)
save(leaders_per_country)

## 5. Tidy things up in a stand-alone python script

Congratulations! You now have a working scraper! However, your code is scattered throughout this notebook along side the tutorials. Hardly production ready...

Copy and paste what you need in a separate `leaders_scraper.py` file.
Make sure it works by calling `python3 leaders_scraper.py`

## (Optional) To go further

If you want to practice scraping, you can read this section and tackle the exercises.

1. Restructure your code by using OOP (see ReadMe).
2. You have noticed the API returns very partial results for country leaders. Many are missing. Overwrite the `get_leaders()` function to get its list from Wikipedia and extract their *personal details* from the frame on the side.

Good luck!